# Zero-shot Generation

## Overview

This notebook handles the first stage of the zero-shot pipeline. Its job is straightforward: take a set of prompts, run the model on them exactly as they are and save the raw outputs.

All prompts are collected from a single text file, keeping generation simple and making it easy to expand or modify the set of tested parameters without touching the notebook logic.

All generated samples are then evaluated in the separate notebook 02_zero_shot_evaluate.ipynb, where we check compilation validity, musical structure, adherence to constraints, and other metrics. Separating evaluation into its own notebook means we don’t have to regenerate outputs every time we update or improve our evaluation logic. The generations stay fixed and reproducible, while the evaluation pipeline can evolve independently.

## Objectives

- Provide a reproducible zero-shot generation workflow.

- Run the model on arbitrary prompts and store every response.

- Produce structured output files that will be consumed and evaluated later in 02_zero_shot_evaluate.ipynb.


## Notebook Structure

This notebook is organized into sections that:
1. Load and configure prompt to be tested

2. Initialize the OpenAI client for HuggingFace router models

3. Generate LilyPond code using the chosen model and parameters

4. Save all generations in a consistent directory structure

5. Record metadata for reproducibility and later evaluation

## Requirements

- **Python 3.8+**
- **HF_TOKEN**: HuggingFace API token (set as environment variable)
- **Model Access**: Access to the selected model via HuggingFace router

## Key Parameters

- **Model** – which model to query (e.g., GPT-OSS-20B)

- **Temperature** – randomness control (e.g., 0.7 for moderate variability)

- **Max Tokens** – upper limit of the generated sequence

- **Number of Runs** – how many samples to generate for each prompt


## Output

All raw generations are saved under:

`zero_shot_outputs/raw`

These raw outputs are then loaded, parsed, and evaluated in the next stage (02_zero_shot_evaluate.ipynb), which handles compilation checks, musical validity metrics, and summary tables.

In [ ]:
from openai import OpenAI
from IPython.display import display, Markdown
from pathlib import Path
from datetime import datetime
import os, re, json


# Helper: extract LilyPond code only
def extract_lilypond_code(text: str) -> str:
    """Extract LilyPond code from a model reply, removing markdown fences/preambles."""
    text = re.sub(r'```(?:lilypond)?\s*\n?(.*?)```', r'\1', text, flags=re.DOTALL)
    text = re.sub(r'^(?:Here|Here\'s|Here is|Output|LilyPond code:|The following).*?:\s*',
                  '', text, flags=re.IGNORECASE)
    return text.strip()


# Generator (saves in zero_shot_outputs/raw/<subfolder>)
def run_generate(
    prompt_file: str | Path = "../configs/prompts/zero_shot.txt",
    *,
    model: str = "openai/gpt-oss-20b:fireworks-ai",
    num_runs: int = 2,
    max_tokens: int = 4096,
    temperature: float = 0.7,
    output_folder_name: str = "session_01"
):

    prompt_path = Path(prompt_file)
    if not prompt_path.exists():
        display(Markdown(f"**❌ Error:** Prompt file not found: `{prompt_path}`"))
        return []

    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        display(Markdown("**❌ Error:** `HF_TOKEN` environment variable not set!"))
        return []

    client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_token)

    # Detect base directory safely (handles notebooks too)
    try:
        base_dir = Path(__file__).resolve().parent
    except NameError:
        base_dir = Path.cwd()  # fallback for Jupyter / REPL

    save_dir = base_dir / "zero_shot_outputs" / "raw" / output_folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    prompt_text = prompt_path.read_text(encoding="utf-8").strip()
    display(Markdown(f"### 📝 Input Prompt\n```\n{prompt_text}\n```\n---"))

    all_results = []
    for i in range(1, num_runs + 1):
        display(Markdown(f"## 🔄 Run {i}/{num_runs}"))
        t0 = datetime.now()
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt_text}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        dt = (datetime.now() - t0).total_seconds()

        choice = resp.choices[0]
        raw_text = (choice.message.content or "") if choice.message else ""
        lp = extract_lilypond_code(raw_text)

        # Display output (no checks)
        md = f"### Model Output\n```lilypond\n{lp}\n```\n"
        md += f"⏱️ Time: {dt:.2f}s | Tokens: {resp.usage.total_tokens if resp.usage else '-'}"
        display(Markdown(md))
        display(Markdown("---"))

        # Save both raw and extracted
        (save_dir / f"raw_{i:02d}.txt").write_text(raw_text, encoding="utf-8")
        (save_dir / f"out_{i:02d}.ly").write_text(lp, encoding="utf-8")

        # Append metadata for summary.json
        all_results.append({
            "run": i,
            "elapsed_seconds": round(dt, 3),
            "tokens_used": resp.usage.total_tokens if resp.usage else None,
            "model": model,
            "temperature": temperature,
            "prompt_file": str(prompt_path)
        })

    # Write summary.json
    summary_path = save_dir / "summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    display(Markdown(f"✅ Saved all outputs and summary in `{save_dir}`"))
    return all_results

# Example:
# results = run_generate(
#     "../configs/prompts/zero_shot.txt",
#     num_runs=3,
#     output_folder_name="test_batch_A"
# )

------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Key</h3>
</div>

##  C Major

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.
Use only lowercase notes a–g with explicit durations (4). Every note must include an octave mark relative to middle C (e.g., c' d' e' f').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.
No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no \transpose, no variables/macros, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.

In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="c_major" )

-----------------

##  F Major

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key f \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key; write required accidentals with `is`/`es` (e.g., bes').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.



In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="f_major" )

-----------

##   E Minor

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key e \minor
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key; write required accidentals with `is`/`es` (e.g., fis').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.



In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="e_minor" )

-----------

##   D Dorian

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key d \dorian
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key; write required accidentals with `is`/`es` if needed.
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.



In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="d_dorian" )

-----------

##  G Mixolydian

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key g \mixolydian
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key; write required accidentals with `is`/`es` if needed.
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="g_mixolydian" )

-----------

##  G Major (1♯)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key g \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key; write required accidentals with `is`/`es` (e.g., fis').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="g_major" )

----------------

##  D Major (2♯)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key d \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key; write required accidentals with `is`/`es` (e.g., fis', cis').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="d_major" )

----------------

##  B♭ Major (2♭)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key bes \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key; write required accidentals with `is`/`es` (e.g., bes', es').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="bes_major" )

----------------

##  A Minor (0)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key a \minor
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key (natural minor/Aeolian); accidentals only if strictly required.
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="a_minor" )

----------------

##  D Minor (1♭)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key d \minor
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared key (natural minor/Aeolian); write required accidentals with `is`/`es` (e.g., bes').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="d_minor" )

----------------

##  C Lydian (modal, 1♯)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \lydian
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Use only notes diatonic to the declared mode; write required accidentals with `is`/`es` (e.g., fis').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.



In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="c_lydian" )

----------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Tempo</h3>
</div>

##   Slow Tempo

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 4/4
\tempo 4 = 60
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="slow_tempo" )

-----------

##   Fast Tempo

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 4/4
\tempo 4 = 160
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly four quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="fast_tempo" )

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Time Signature</h3>
</div>

##  3/4 (simple triple)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 3/4
\tempo 4 = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (4). Every note must include an octave mark (e.g., c' d' e' f').
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly three quarter-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="three_four" )

-------------------

##  6/8 (compound duple)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 6/8
\tempo 4. = 100
Do not use \relative anywhere.

Use LilyPond pitch names with explicit durations (8). Every note must include an octave mark (e.g., c'8 d'8 e'8 f'8).
Keep all pitches within the inclusive range c'..c''.
Write exactly 8 bars, each bar containing exactly six eighth-notes, separated by |, and end the last bar with |.

No rests, no chords, no repeats, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="six_eight" )

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Harmony </h3>
</div>

##  Dyads (2-note chords)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Write only dyads: every sounded event must be a chord with exactly two simultaneous notes, using LilyPond chord syntax < ... > and explicit durations (4), e.g., <c' e'>4.
Use only LilyPond pitch names; every note must include an octave mark (e.g., c', d', e', f').
Use only notes diatonic to the declared key; if an accidental is required, use `is`/`es` (e.g., fis', bes').
Keep all chord tones within the inclusive range c'..c'' (both notes of each dyad must satisfy this).
Write exactly 8 bars, each bar containing exactly four quarter-note dyads, separated by |, and end the last bar with |.

No single notes, no rests, no chords with more or fewer than two notes, no arpeggios, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no << >> or \\ voices, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="dyads" )

------------

##  Triads (3-note chords)

prompt:
Write LilyPond code only.
Put \version "2.24.4" on the first line.
Then write exactly one top-level music block enclosed in { ... } (no other top-level music).
At the start of that block, include exactly these directives, in this order and with no blank lines or comments between them:
\clef treble
\key c \major
\time 4/4
\tempo 4 = 100
Do not use \relative anywhere.

Write only triads: every sounded event must be a chord with exactly three simultaneous notes, using LilyPond chord syntax < ... > and explicit durations (4), e.g., <c' e' g'>4.
Use only LilyPond pitch names; every note must include an octave mark (e.g., c', d', e', f').
Use only notes diatonic to the declared key; if an accidental is required, use `is`/`es` (e.g., fis', bes').
Keep all chord tones within the inclusive range c'..c'' (all three notes of each triad must satisfy this).
Write exactly 8 bars, each bar containing exactly four quarter-note triads, separated by |, and end the last bar with |.

No single notes, no rests, no chords with fewer or more than three notes, no arpeggios, no ties (~), no slurs ( ), no articulations, no dynamics, no tuplets, no transpositions, no variables/macros, no music functions, no text/markup, no << >> or \\ voices, no \paper, no \header, no \layout, no \score, and nothing before or after the single block.


In [ ]:
results = run_generate(     "../../configs/prompts/zero_shot.txt",     num_runs=20,     output_folder_name="triads" )